# VQ-VAE Training (standalone)
Trains VQ-VAE for 20 epochs and pushes checkpoint to HF Hub.

In [ ]:
import os, sys, torch
!pip install -q datasets huggingface_hub torchvision pillow tqdm
!git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_DATASET = os.environ.get("HF_DATASET", "darklord8777/sprites")
HF_MODEL_REPO = "darklord8777/sprite-generator-model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"HF_TOKEN set: {bool(HF_TOKEN)}")

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm

class SpriteDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset_path, split="train", image_size=32):
        self.dataset = load_dataset(hf_dataset_path, split=split)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img = self.dataset[idx]["image"].convert("RGBA")
        return self.transform(img)

dataset = SpriteDataset(HF_DATASET)
loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2)
print(f"Dataset: {len(dataset)} sprites")

In [ ]:
from pathlib import Path
import torch.nn as nn
from models.vqvae.model import VQVAE

model = VQVAE(in_channels=4, hidden_dim=128, latent_dim=64, num_embeddings=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
checkpoint_dir = Path("/kaggle/working/checkpoints/vqvae")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        output = model(batch)
        output["loss"].backward()
        optimizer.step()
        total_loss += output["loss"].item()
        pbar.set_postfix({"loss": output["loss"].item()})
    
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}: loss={avg_loss:.6f}")
    
    if (epoch + 1) % 10 == 0:
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss": avg_loss,
        }, checkpoint_dir / f"vqvae_epoch_{epoch+1:03d}.pt")

torch.save({
    "epoch": EPOCHS - 1,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "loss": avg_loss,
}, checkpoint_dir / "vqvae_final.pt")
print("VQ-VAE training complete")

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_file(
        path_or_fileobj=str(checkpoint_dir / "vqvae_final.pt"),
        path_in_repo="vqvae_latest.pt",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    print("VQ-VAE checkpoint pushed to HF Hub")
else:
    print("No HF_TOKEN set, skipping push")